In [11]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings
import statsmodels.stats.api as ssa
import statsmodels.api as sma
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV,cross_val_score
from sklearn.ensemble import RandomForestRegressor
import os
import joblib

warnings.filterwarnings('ignore')
PIPELINE='pipeline.pkl'
MODEL_FILE='model_file.pkl'
def built_pipeline(num_train,cat_train):
    num_pipeline=Pipeline([('impute',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
    cat_pipeline=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(handle_unknown='ignore'))])
    merge_pipeline=ColumnTransformer([
        ('num_data',num_pipeline,list(num_train.columns)),
        ('cat_data',cat_pipeline,list(cat_train.columns))])
    return merge_pipeline
if not os.path.exists(MODEL_FILE):
    data=pd.read_csv("air_quality_index.csv")
    num_data=data.select_dtypes(include=np.number)
    cat_data=data.select_dtypes(exclude=np.number)
    city_unique=data['city'].unique()
    state_unique=data['state'].unique()
    
    ind_stats=["Andhra Pradesh","Arunachal Pradesh","Assam","Bihar","Chhattisgarh","Goa","Gujarat","Haryana","Himachal Pradesh",
    "Jammu and Kashmir","Jharkhand","Karnataka","Kerala","Madhya Pradesh","Maharashtra","Manipur","Meghalaya","Mizoram","Nagaland",
  "Odisha","Punjab","Rajasthan","Sikkim","Tamil Nadu","Telangana","Tripura","Uttar Pradesh","Uttarakhand","West Bengal"]

    data['pollu']=pd.cut(data['pollutant_avg'],bins=[0,5,10,20,30,40,50,80,120,160,220,np.inf],labels=[1,2,3,4,5,6,7,8,9,10,11])
    num_data_=['latitude', 'longitude', 'pollutant_min', 'pollutant_max']
    data['state']=data['state'].replace(to_replace="Chandigarh",value='Punjab')
    
    data['state']=data['state'].str.replace("_"," ")
    
    data['pollu']=data['pollu'].fillna(data['pollu'].mode()[0])
    y=data['pollu']
    x=data.drop('pollu',axis=1)
    # Here i drop the garbage cat_col
    data=data.drop(['country','last_update','pollu','longitude'],axis=1)

    from sklearn.model_selection import StratifiedShuffleSplit
    splits=StratifiedShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
    for train,test in splits.split(x,y):
        strat_train=data.loc[train]
        strat_test=data.loc[test]
    train=strat_train.copy()
    test=strat_test.copy()

    #Advanced Approch for handling missing values
    pollu_min=train.groupby('state')['pollutant_min'].median()
    pollu_max=train.groupby('state')['pollutant_max'].median()
    pollu_avg=train.groupby('state')['pollutant_avg'].median()
    # Train data missing value
    train['pollutant_max']=train['pollutant_max'].fillna(train['state'].map(pollu_max))
    train['pollutant_min']=train['pollutant_min'].fillna(train['state'].map(pollu_min))
    train['pollutant_avg']=train['pollutant_avg'].fillna(train['state'].map(pollu_avg))
    # Test data missing values
    test['pollutant_max']=test['pollutant_max'].fillna(test['state'].map(pollu_max))
    test['pollutant_min']=test['pollutant_min'].fillna(test['state'].map(pollu_min))
    test['pollutant_avg']=test['pollutant_avg'].fillna(test['state'].map(pollu_avg))

    train=train.drop(['state','city'],axis=1)
    test=test.drop(['state','city'],axis=1)
    train_label=train['pollutant_avg']
    train=train.drop('pollutant_avg',axis=1)
    test_label=test['pollutant_avg']
    test=test.drop('pollutant_avg',axis=1)  
    
    num_train=train.select_dtypes(include=np.number)
    cat_train=train.select_dtypes(exclude=np.number)
    
    # Here we are handling only train data outliers not test data outliers.
    for i in num_train:
        q1,q3=np.quantile(train[i],[0.25,0.95])
        iqr=q3-q1
        ul=q3+(1.5*iqr)
        ll=q1-(1.5*iqr)
        ll,ul=np.quantile(train[i],[0.10,0.90])
        train.loc[train[i]>ul,i]=ul
        train.loc[train[i]<ll,i]=ll

    pipeline=built_pipeline(num_train,cat_train)
    prepared=pipeline.fit_transform(train,train_label)
    prepared_test=pipeline.transform(test)
    
    random=RandomForestRegressor(max_depth=10,n_estimators=100,min_samples_leaf=5,max_leaf_nodes=120,random_state=42,n_jobs=-1)
    random_prepared=random.fit(prepared,train_label)
    prediction=random_prepared.predict(prepared_test)
    
    joblib.dump(random_prepared,MODEL_FILE)
    joblib.dump(pipeline,PIPELINE)
else:
    model=joblib.load(MODEL_FILE)
    pipeline=joblib.load(PIPELINE)
    trans_data=pipeline.transform(strat_test)
    prediction_tr=model.predict(trans_data)
    prediction=pd.DataFrame(prediction_tr,columns=['pollutant_avg'])
    prediction.to_csv('Air-predict-output')
    print("Your Model is ready")
    

Your Model is ready


In [12]:
r2=r2_score(test_label,prediction)
print(r2)

0.8493599083994532
